# AlexNet-LSTM Model for MNIST RT Prediction

This notebook demonstrates how to train and evaluate the AlexNet-LSTM model for MNIST digit classification with reaction time (RT) prediction.

## 1. Setup and Imports

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
import numpy as np
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import os
from torchvision import models

from preprocess_mnist_behavioral import MNISTBehavioralDataset

%matplotlib inline

try:
    plt.style.use('seaborn-v0_8-darkgrid')
except:
    pass
sns.set_palette("husl")

In [ ]:
# Check device
if torch.cuda.is_available():
    device = torch.device('cuda:0')
    print(f"Using CUDA: {torch.cuda.get_device_name(0)}")
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = torch.device('mps')
    print("Using Apple MPS")
else:
    device = torch.device('cpu')
    print("Using CPU")

print(f"Device: {device}")

## 2. Define Model Components

In [ ]:
def add_noise(x, mask_p=0.0, std=0.0, rescale_after_dropout=True):
    if mask_p == 0 and std == 0:
        return x

    x_noisy = x.clone()

    if mask_p > 0:
        mask = torch.bernoulli(torch.ones_like(x) * (1 - mask_p))
        x_noisy = x_noisy * mask
        if rescale_after_dropout:
            x_noisy = x_noisy / (1 - mask_p + 1e-8)

    if std > 0:
        noise = torch.randn_like(x) * std
        x_noisy = x_noisy + noise

    return x_noisy

In [ ]:
class PretrainedAlexNet(nn.Module):
    def __init__(self, feature_dim=4096, freeze_features=True):
        super().__init__()
        
        alexnet = models.alexnet(pretrained=True)
        self.features = alexnet.features
        self.avgpool = alexnet.avgpool
        
        self.classifier = nn.Sequential(
            nn.Linear(256 * 6 * 6, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(4096, feature_dim),
            nn.ReLU(inplace=True),
        )
        
        if freeze_features:
            for param in self.features.parameters():
                param.requires_grad = False
        
        self.feature_dim = feature_dim
    
    def forward(self, x):
        if x.size(1) == 1:
            x = x.repeat(1, 3, 1, 1)
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

In [ ]:
class DiffDecision(torch.autograd.Function):
    @staticmethod
    def forward(ctx, trajectory, dsdt_trajectory):
        mask = trajectory > 0
        decision_time = mask.float().argmax(dim=1).float()
        decision_time[mask.sum(dim=1) == 0] = float(trajectory.shape[1] - 1)
        ctx.save_for_backward(dsdt_trajectory, decision_time, trajectory)
        return decision_time

    @staticmethod
    def backward(ctx, grad_output):
        dsdt_trajectory, decision_times, trajectory = ctx.saved_tensors
        device = dsdt_trajectory.device
        mask = trajectory > 0
        idx1 = (mask.sum(dim=1) == 0)
        idx2 = dsdt_trajectory[torch.arange(dsdt_trajectory.size(0), device=device), decision_times.long()] < 0
        idx = torch.logical_and(idx1, idx2)
        grads = torch.zeros_like(dsdt_trajectory)
        batch_indices = torch.arange(decision_times.size(0), device=device)
        grads[batch_indices, decision_times.long()] = -1.0 / (dsdt_trajectory[batch_indices, decision_times.long()] + 1e-6)
        grads[batch_indices[idx], decision_times[idx].long()] = 1e-6
        grads = grads * grad_output.unsqueeze(1)
        return grads, None

In [ ]:
class RTify_LSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_size,
                 time_steps=20, sigma=2.0,
                 noise_position='input',
                 mask_p=0.0, gaussian_std=0.0,
                 evidence_noise_std=0.0, evidence_mask_p=0.0,
                 evidence_dropout_rescale=False,
                 evidence_scale=1.0,
                 threshold=6.0,
                 num_lstm_layers=1):

        super().__init__()
        self.evidence_dropout_rescale = evidence_dropout_rescale
        self.time_steps = time_steps
        self.noise_position = noise_position
        self.mask_p = mask_p
        self.gaussian_std = gaussian_std
        self.evidence_noise_std = evidence_noise_std
        self.evidence_mask_p = evidence_mask_p
        self.evidence_scale = evidence_scale
        self.hidden_dim = hidden_dim

        self.input_norm = nn.LayerNorm(input_dim)

        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_lstm_layers,
            batch_first=False
        )

        self.fc = nn.Linear(hidden_dim, output_size)

        self.evidence = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
            nn.Tanh()
        )
        self.register_buffer('threshold', torch.tensor(threshold))
        self.sigma = sigma

    def forward(self, x):
        B, input_dim = x.shape
        x_normed = self.input_norm(x)
        x_seq = x_normed.unsqueeze(0).repeat(self.time_steps, 1, 1)

        if self.noise_position in ['input', 'both']:
            x_seq = add_noise(x_seq, self.mask_p, self.gaussian_std)

        hidden_states, _ = self.lstm(x_seq)

        logit_trajectory = self.fc(hidden_states).permute(1, 0, 2)
        s_traj = self.evidence(hidden_states).squeeze(-1).permute(1, 0) * self.evidence_scale

        if self.noise_position in ['evidence', 'both']:
            s_traj = add_noise(
                s_traj,
                self.evidence_mask_p,
                self.evidence_noise_std,
                rescale_after_dropout=self.evidence_dropout_rescale
            )

        s_accumulated = torch.cumsum(s_traj, dim=1)
        dsdt_trajectory = torch.diff(s_accumulated, dim=1)
        dsdt_trajectory = torch.cat([dsdt_trajectory[:, :1], dsdt_trajectory], dim=1)

        decision_time = DiffDecision.apply(s_accumulated - self.threshold, dsdt_trajectory)

        soft_index = torch.exp(-0.5 * (decision_time.unsqueeze(1) -
                               torch.arange(self.time_steps, device=x.device)) ** 2 / self.sigma ** 2)
        soft_index = soft_index / soft_index.sum(dim=-1, keepdim=True)
        decision_logits = (logit_trajectory * soft_index.unsqueeze(-1)).sum(dim=1)

        return decision_logits, (decision_time + 1) / self.time_steps

In [ ]:
class AlexNetRTifyModel(nn.Module):
    def __init__(self, feature_dim=4096, hidden_dim=512, output_size=10,
                 time_steps=20, sigma=2.0, freeze_encoder=True,
                 noise_position='evidence',
                 mask_p=0.0, gaussian_std=0.0,
                 evidence_noise_std=0.0, evidence_mask_p=0.0,
                 evidence_dropout_rescale=False, evidence_scale=1.0,
                 threshold=6.0, num_lstm_layers=1):
        super().__init__()
        self.freeze_encoder = freeze_encoder

        self.encoder = PretrainedAlexNet(feature_dim=feature_dim, freeze_features=freeze_encoder)

        self.rtify = RTify_LSTM(
            input_dim=feature_dim,
            hidden_dim=hidden_dim,
            output_size=output_size,
            time_steps=time_steps,
            sigma=sigma,
            noise_position=noise_position,
            mask_p=mask_p,
            gaussian_std=gaussian_std,
            evidence_noise_std=evidence_noise_std,
            evidence_mask_p=evidence_mask_p,
            evidence_dropout_rescale=evidence_dropout_rescale,
            evidence_scale=evidence_scale,
            threshold=threshold,
            num_lstm_layers=num_lstm_layers
        )

    def forward(self, image):
        z = self.encoder(image)
        decision_logits, decision_time = self.rtify(z)
        return decision_logits, decision_time, z

## 3. Load Data

In [ ]:
# Configuration
DATA_PATH = 'RTNet_Dataset/behavioral data.csv'
BATCH_SIZE = 32
TEST_SPLIT = 0.2
RANDOM_SEED = 42

# Set random seed
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

In [ ]:
# Load dataset
print("Loading dataset...")
full_dataset = MNISTBehavioralDataset(DATA_PATH)

# Split into train and test
total_len = len(full_dataset)
train_size = int((1 - TEST_SPLIT) * total_len)
test_size = total_len - train_size

train_dataset, test_dataset = torch.utils.data.random_split(
    full_dataset, [train_size, test_size],
    generator=torch.Generator().manual_seed(RANDOM_SEED)
)

# Create data loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")

In [ ]:
# Visualize a sample
sample = full_dataset[0]
print("\nSample data:")
print(f"  Image shape: {sample['image'].shape}")
print(f"  Label: {sample['label'].item()}")
print(f"  RT (normalized): {sample['rt_normalized'].item():.4f}")
print(f"  RT (original): {sample['rt_original'].item():.4f} seconds")
print(f"  Correct: {sample['correct'].item()}")

plt.figure(figsize=(4, 4))
plt.imshow(sample['image'].squeeze(), cmap='gray')
plt.title(f"Label: {sample['label'].item()}, RT: {sample['rt_original'].item():.3f}s")
plt.axis('off')
plt.show()

## 4. Create Model

In [ ]:
# Model configuration
config = {
    'feature_dim': 4096,
    'hidden_dim': 512,
    'output_size': 10,
    'time_steps': 20,
    'freeze_encoder': True,
    'noise_position': 'evidence',
    'evidence_noise_std': 0.5,
    'evidence_mask_p': 0.4,
    'threshold': 6.0,
    'evidence_scale': 1.0,
    'num_lstm_layers': 1
}

# Create model
model = AlexNetRTifyModel(**config)
model = model.to(device)

# Print model info
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

## 5. Training

In [ ]:
# Training configuration
NUM_EPOCHS = 10
LEARNING_RATE = 1e-4
USE_RT_LOSS = True  # Set to True to use RT supervision
SPEED_PENALTY = 0.0

# Loss functions and optimizer
label_criterion = nn.CrossEntropyLoss()
rt_criterion = nn.MSELoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LEARNING_RATE)

# Training history
history = {
    'train_loss': [],
    'train_acc': [],
    'train_rt_loss': [],
    'test_acc': [],
    'test_corr': []
}

In [ ]:
# Training loop
print("Starting training...")
print(f"RT Supervision: {USE_RT_LOSS}")
print("="*60)

for epoch in range(NUM_EPOCHS):
    # Training phase
    model.train()
    epoch_loss = 0.0
    epoch_acc = 0.0
    epoch_rt_loss = 0.0
    num_batches = 0
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}")
    for batch in pbar:
        images = batch['image'].to(device)
        labels = batch['label'].to(device)
        rt = batch['rt_normalized'].to(device)
        
        optimizer.zero_grad()
        decision_logits, rt_pred, _ = model(images)
        
        # Calculate losses
        label_loss = label_criterion(decision_logits, labels)
        rt_loss = rt_criterion(rt_pred, rt)
        
        if USE_RT_LOSS:
            total_loss = label_loss + rt_loss + SPEED_PENALTY * rt_pred.mean()
        else:
            total_loss = label_loss + SPEED_PENALTY * rt_pred.mean()
        
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        # Metrics
        acc = (decision_logits.argmax(-1) == labels).float().mean().item()
        epoch_loss += total_loss.item()
        epoch_acc += acc
        epoch_rt_loss += rt_loss.item()
        num_batches += 1
        
        pbar.set_postfix({'loss': f'{total_loss.item():.4f}', 'acc': f'{acc:.3f}'})
    
    # Record training metrics
    history['train_loss'].append(epoch_loss / num_batches)
    history['train_acc'].append(epoch_acc / num_batches)
    history['train_rt_loss'].append(epoch_rt_loss / num_batches)
    
    # Evaluation phase
    model.eval()
    all_rt_pred = []
    all_rt_human = []
    correct = 0
    total = 0
    
    with torch.no_grad():
        for batch in test_loader:
            images = batch['image'].to(device)
            labels = batch['label'].to(device)
            rt_human = batch['rt_normalized'].to(device)
            
            decision_logits, decision_time, _ = model(images)
            pred_labels = decision_logits.argmax(dim=-1)
            
            correct += (pred_labels == labels).sum().item()
            total += labels.size(0)
            
            all_rt_pred.extend(decision_time.cpu().numpy())
            all_rt_human.extend(rt_human.cpu().numpy())
    
    test_acc = correct / total
    correlation = np.corrcoef(all_rt_pred, all_rt_human)[0, 1]
    
    history['test_acc'].append(test_acc)
    history['test_corr'].append(correlation)
    
    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}:")
    print(f"  Train Loss: {history['train_loss'][-1]:.4f}")
    print(f"  Train Acc: {history['train_acc'][-1]*100:.2f}%")
    print(f"  Test Acc: {test_acc*100:.2f}%")
    print(f"  RT Correlation: {correlation:.4f}")
    print("-"*60)

print("\nTraining complete!")

## 6. Visualize Training Results

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Loss
axes[0].plot(history['train_loss'], label='Train Loss', marker='o')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')
axes[0].legend()
axes[0].grid(True)

# Accuracy
axes[1].plot([x*100 for x in history['train_acc']], label='Train', marker='o')
axes[1].plot([x*100 for x in history['test_acc']], label='Test', marker='s')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Accuracy')
axes[1].legend()
axes[1].grid(True)

# RT Correlation
axes[2].plot(history['test_corr'], label='RT Correlation', marker='o', color='green')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Correlation')
axes[2].set_title('RT Correlation (Test Set)')
axes[2].legend()
axes[2].grid(True)

plt.tight_layout()
plt.show()

## 7. Final Evaluation

In [ ]:
# Detailed evaluation
model.eval()

all_rt_pred = []
all_rt_human = []
all_labels = []
all_preds = []
all_correct = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Evaluating"):
        images = batch['image'].to(device)
        labels = batch['label'].to(device)
        rt_human = batch['rt_normalized'].to(device)
        correct = batch['correct'].to(device)
        
        decision_logits, decision_time, _ = model(images)
        pred_labels = decision_logits.argmax(dim=-1)
        
        all_rt_pred.extend(decision_time.cpu().numpy())
        all_rt_human.extend(rt_human.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(pred_labels.cpu().numpy())
        all_correct.extend(correct.cpu().numpy())

all_rt_pred = np.array(all_rt_pred)
all_rt_human = np.array(all_rt_human)
all_labels = np.array(all_labels)
all_preds = np.array(all_preds)
all_correct = np.array(all_correct)

# Calculate metrics
accuracy = np.mean(all_preds == all_labels)
correlation = np.corrcoef(all_rt_pred, all_rt_human)[0, 1]

correct_rt = all_rt_pred[all_correct == 1]
incorrect_rt = all_rt_pred[all_correct == 0]

print("\n" + "="*60)
print("Final Test Results")
print("="*60)
print(f"Accuracy: {accuracy*100:.2f}%")
print(f"RT Correlation: {correlation:.4f}")
print(f"\nRT by Correctness (normalized):")
print(f"  Correct trials: {correct_rt.mean():.4f} ± {correct_rt.std():.4f} (n={len(correct_rt)})")
print(f"  Incorrect trials: {incorrect_rt.mean():.4f} ± {incorrect_rt.std():.4f} (n={len(incorrect_rt)})")

# Denormalize RT
correct_rt_seconds = full_dataset.denormalize_rt(correct_rt)
incorrect_rt_seconds = full_dataset.denormalize_rt(incorrect_rt)
print(f"\nRT by Correctness (seconds):")
print(f"  Correct trials: {correct_rt_seconds.mean():.3f} ± {correct_rt_seconds.std():.3f} s")
print(f"  Incorrect trials: {incorrect_rt_seconds.mean():.3f} ± {incorrect_rt_seconds.std():.3f} s")

In [ ]:
# Plot RT distributions
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Model RT vs Human RT
axes[0].scatter(all_rt_human, all_rt_pred, alpha=0.3, s=10)
axes[0].set_xlabel('Human RT (normalized)')
axes[0].set_ylabel('Model RT (normalized)')
axes[0].set_title(f'Model vs Human RT (r={correlation:.3f})')
axes[0].grid(True, alpha=0.3)

# Add correlation line
z = np.polyfit(all_rt_human, all_rt_pred, 1)
p = np.poly1d(z)
axes[0].plot(all_rt_human, p(all_rt_human), "r--", alpha=0.8, label=f'fit line')
axes[0].legend()

# RT distribution by correctness
axes[1].hist(correct_rt, bins=30, alpha=0.6, label=f'Correct (n={len(correct_rt)})', density=True)
axes[1].hist(incorrect_rt, bins=30, alpha=0.6, label=f'Incorrect (n={len(incorrect_rt)})', density=True)
axes[1].set_xlabel('RT (normalized)')
axes[1].set_ylabel('Density')
axes[1].set_title('RT Distribution by Correctness')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Confusion matrix
from sklearn.metrics import confusion_matrix, classification_report

cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=range(10), yticklabels=range(10))
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()

print("\nClassification Report:")
print(classification_report(all_labels, all_preds, digits=4))

## 8. Save Model

In [ ]:
# Save model
OUTPUT_DIR = './output_mnist'
os.makedirs(OUTPUT_DIR, exist_ok=True)

model_path = os.path.join(OUTPUT_DIR, 'alexnet_lstm_mnist.pth')
torch.save({
    'model_state_dict': model.state_dict(),
    'config': config,
    'history': history,
    'final_accuracy': accuracy,
    'final_correlation': correlation
}, model_path)

print(f"Model saved to: {model_path}")

# Save results
results_df = pd.DataFrame({
    'true_label': all_labels,
    'pred_label': all_preds,
    'correct': all_correct,
    'rt_pred_normalized': all_rt_pred,
    'rt_human_normalized': all_rt_human,
    'rt_pred_seconds': full_dataset.denormalize_rt(all_rt_pred),
    'rt_human_seconds': full_dataset.denormalize_rt(all_rt_human)
})

results_path = os.path.join(OUTPUT_DIR, 'results.csv')
results_df.to_csv(results_path, index=False)
print(f"Results saved to: {results_path}")

## 9. Load and Test Saved Model (Optional)

In [ ]:
# Load saved model
checkpoint = torch.load(model_path, map_location=device)

# Create new model with saved config
loaded_model = AlexNetRTifyModel(**checkpoint['config'])
loaded_model.load_state_dict(checkpoint['model_state_dict'])
loaded_model = loaded_model.to(device)
loaded_model.eval()

print(f"Loaded model from epoch with accuracy: {checkpoint['final_accuracy']*100:.2f}%")
print(f"RT Correlation: {checkpoint['final_correlation']:.4f}")

In [ ]:
# Test on a single sample
sample = full_dataset[100]
image = sample['image'].unsqueeze(0).to(device)
true_label = sample['label'].item()
true_rt = sample['rt_original'].item()

with torch.no_grad():
    logits, rt_pred, _ = loaded_model(image)
    pred_label = logits.argmax(dim=-1).item()
    pred_rt_normalized = rt_pred.item()
    pred_rt_seconds = full_dataset.denormalize_rt(pred_rt_normalized)

print(f"\nSample Test:")
print(f"  True label: {true_label}")
print(f"  Predicted label: {pred_label}")
print(f"  Correct: {pred_label == true_label}")
print(f"  True RT: {true_rt:.3f} seconds")
print(f"  Predicted RT: {pred_rt_seconds:.3f} seconds")

plt.figure(figsize=(3, 3))
plt.imshow(sample['image'].squeeze(), cmap='gray')
plt.title(f"True: {true_label}, Pred: {pred_label}\nRT: {pred_rt_seconds:.3f}s")
plt.axis('off')
plt.show()